# Alerts Signals — Monitoring

Reads from `ALERTS_SIGNALS` and produces summary stats on detection output.

| Cell | What happens |
|------|--------------|
| 1 | Config — source table, optional date range filter |
| 2 | Session — connect to Snowflake |
| 3 | Summary — signal counts by type and trigger over time |
| 4 | Store view — per-store breakdown of signal type and trigger |

In [1]:
# ── Source table ───────────────────────────────────────────────────────────────
SIGNALS_TABLE = 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS'

# ── Optional date filter (set to None for all available dates) ─────────────────
FROM_DATE = None   # e.g. '2026-01-01'
TO_DATE   = None   # e.g. '2026-03-31'

# ── Optional store filter (set to None for all stores) ────────────────────────
STORE_ID  = None   # e.g. 507

print(f'Source table : {SIGNALS_TABLE}')
print(f'Date filter  : {FROM_DATE or "all"} → {TO_DATE or "all"}')
print(f'Store filter : {STORE_ID or "all stores"}')

Source table : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS
Date filter  : all → all
Store filter : all stores


In [2]:
import sys, os
import pandas as pd

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import store_insight.pipeline
from store_insight.pipeline.pipeline.session import load_env, create_session

load_env()
session = create_session()
print('Snowflake session created.')

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJLb%2BIwFIX%2FSuRZJ7ETXrGAipapBtF2EITRdHYmNmDhR2o7hPLrxwkgdRbtYjZRZJ97v%2BN77vDuJEVwZMZyrUYARRAETBWacrUbgXX%2BGA5AYB1RlAit2Ai8MwvuxkNLpCjxpHJ7tWRvFbMu8I2Uxc3FCFRGYU0st1gRySx2BV5Nnp9wEkFMrGXGeRy4llDLPWvvXInjuK7rqE4jbXZxAiGMYRZ7VSP5Bj4gyq8ZpdFOF1rcSk7%2BTZ8gUAw7DcIrPGFxLbzn6jKCryibi8jiH3m%2BCBc%2FVzkIJrfXPWhlK8nMipkjL9h6%2BXQxYL0DK1EvhZ2oOoRW%2BxlG5FwZFlml660gB1ZoWVbOt478X7xlNBZ6x%2F3AZtMRKA%2BcvmZ8KU4rOVD93Wb%2F%2B62S2Vomz%2BdurubT8%2Bv8SKicl1Xv%2FvuhAMGvW7xJE%2B%2FM2orNVBOq80cw6YWwE8IkRxCjDCdplPb7f0Aw9aFyRVxbeXPe%2BogkL4y2euu0Elyx1mWWkhTRLg0R8Z9Omm7DDSEwROmg28kKmiA4iJvoEnBZH9waMeP%2FGsow%2Ftjiuo4vPqHZdKEFL96DR20kcZ8HiCLUnnAablspZpJwMaHUMGt9kELo%2BsEw4v

 pip install snowflake-connector-python[secure-local-storage]


In [6]:
# ── Load & filter ──────────────────────────────────────────────────────────────
from snowflake.snowpark import functions as F

df = session.table(SIGNALS_TABLE)

if FROM_DATE:
    df = df.filter(F.col('RUN_DATE') >= FROM_DATE)
if TO_DATE:
    df = df.filter(F.col('RUN_DATE') <= TO_DATE)
if STORE_ID:
    df = df.filter(F.col('STORE_ID') == STORE_ID)

total = df.count()
dates = df.select(F.min('RUN_DATE').alias('FIRST'), F.max('RUN_DATE').alias('LAST'), F.countDistinct('RUN_DATE').alias('N_DATES')).collect()[0]
print(f'Total signal rows : {total:,}')
print(f'Run dates         : {dates["FIRST"]} → {dates["LAST"]}  ({dates["N_DATES"]} dates)')

# ── Signals per run date ───────────────────────────────────────────────────────
by_date = (
    df.group_by('RUN_DATE', 'SIGNAL_TYPE')
    .agg(F.count('*').alias('SIGNAL_COUNT'))
    .sort('RUN_DATE', 'SIGNAL_TYPE')
    .to_pandas()
)
print('\n── Signals per run date ──')
display(by_date.pivot(index='RUN_DATE', columns='SIGNAL_TYPE', values='SIGNAL_COUNT').fillna(0).astype(int))

# ── Signals by signal type (totals) ───────────────────────────────────────────
by_type = (
    df.group_by('SIGNAL_TYPE')
    .agg(F.count('*').alias('TOTAL'), F.countDistinct('RUN_DATE').alias('DATES_SEEN'))
    .sort('TOTAL', ascending=False)
    .to_pandas()
)
print('\n── Totals by signal type ──')
display(by_type)

# ── Explode TRIGGER_TYPES (comma-separated) via LATERAL FLATTEN ───────────────
# "TRIGGER" is a reserved word in Snowflake — alias it as TRIGGER_NAME
df.create_or_replace_temp_view('_signals_tmp')
exploded = session.sql("""
    SELECT
        RUN_DATE,
        SIGNAL_TYPE,
        STORE_ID,
        TRIM(t.value::STRING) AS TRIGGER_NAME
    FROM _signals_tmp,
    LATERAL FLATTEN(input => SPLIT(TRIGGER_TYPES, ',')) t
    WHERE TRIGGER_TYPES IS NOT NULL
""")

# ── Totals by trigger ──────────────────────────────────────────────────────────
by_trigger = (
    exploded.group_by('SIGNAL_TYPE', 'TRIGGER_NAME')
    .agg(
        F.count('*').alias('TOTAL'),
        F.countDistinct('RUN_DATE').alias('DATES_SEEN'),
        F.countDistinct('STORE_ID').alias('STORES_AFFECTED'),
    )
    .sort('SIGNAL_TYPE', 'TOTAL', ascending=[True, False])
    .to_pandas()
)
print('\n── Totals by trigger ──')
display(by_trigger)

# ── Triggers per run date (trend) ─────────────────────────────────────────────
trigger_trend = (
    exploded.group_by('RUN_DATE', 'SIGNAL_TYPE', 'TRIGGER_NAME')
    .agg(F.count('*').alias('COUNT'))
    .sort('RUN_DATE', 'SIGNAL_TYPE', 'TRIGGER_NAME')
    .to_pandas()
)
print('\n── Trigger counts per run date ──')
display(
    trigger_trend.pivot(index='RUN_DATE', columns=['SIGNAL_TYPE', 'TRIGGER_NAME'], values='COUNT').fillna(0).astype(int)
)

Total signal rows : 5,771
Run dates         : 2026-03-01 → 2026-03-16  (16 dates)

── Signals per run date ──


SIGNAL_TYPE,COMPLIANCE,OSA
RUN_DATE,,
2026-03-01,14,329
2026-03-02,13,329
2026-03-03,15,332
2026-03-04,17,333
2026-03-05,23,346
2026-03-06,22,348
2026-03-07,15,359
2026-03-08,14,358
2026-03-09,13,363



── Totals by signal type ──


,SIGNAL_TYPE,TOTAL,DATES_SEEN
0,OSA,5533,16
1,COMPLIANCE,238,16



── Totals by trigger ──


,SIGNAL_TYPE,TRIGGER_NAME,TOTAL,DATES_SEEN,STORES_AFFECTED
0,COMPLIANCE,TICK_AND_FLICK,238,16,63
1,OSA,WOW,3553,16,117
2,OSA,VS_CHAIN,2251,16,109
3,OSA,REPEAT,2066,16,81



── Trigger counts per run date ──


SIGNAL_TYPE      COMPLIANCE    OSA              
TRIGGER_NAME TICK_AND_FLICK REPEAT VS_CHAIN  WOW
RUN_DATE                                        
2026-03-01               14    131      139  190
2026-03-02               13    129      140  201
2026-03-03               15    130      145  202
2026-03-04               17    128      145  207
2026-03-05               23    130      144  222
2026-03-06               22    131      140  220
2026-03-07               15    129      143  240
2026-03-08               14    132      138  233
2026-03-09               13    130      137  240
2026-03-10               14    126      137  231
2026-03-11               14    129      139  233
2026-03-12               11    130      143  233
2026-03-13               16    129      138  242
2026-03-14               13    127      146  231
2026-03-15               11    129      141  214
2026-03-16               13    126      136  214

In [8]:
# ── Per-store breakdown ────────────────────────────────────────────────────────
# Uses the exploded view created in cell 3 (_signals_tmp / exploded)

store_summary = (
    exploded.group_by('STORE_ID', 'SIGNAL_TYPE', 'TRIGGER_NAME')
    .agg(
        F.count('*').alias('TOTAL_SIGNALS'),
        F.countDistinct('RUN_DATE').alias('WEEKS_FIRED'),
    )
    .sort('STORE_ID', 'SIGNAL_TYPE', 'TRIGGER_NAME')
    .to_pandas()
)

# Wide pivot: stores as rows, (signal_type, trigger) as columns
store_wide = store_summary.pivot(
    index='STORE_ID',
    columns=['SIGNAL_TYPE', 'TRIGGER_NAME'],
    values='TOTAL_SIGNALS',
).fillna(0).astype(int)

store_wide['_TOTAL'] = store_wide.sum(axis=1)
store_wide = store_wide.sort_values('_TOTAL', ascending=False).drop(columns='_TOTAL')

print(f'Stores with at least one signal: {len(store_wide)}')
print('\n── Signals per store (total across all run dates) ──')
display(store_wide)

# ── Weeks fired per store ──────────────────────────────────────────────────────
store_weeks = store_summary.pivot(
    index='STORE_ID',
    columns=['SIGNAL_TYPE', 'TRIGGER_NAME'],
    values='WEEKS_FIRED',
).fillna(0).astype(int)

store_weeks['_TOTAL'] = store_weeks.sum(axis=1)
store_weeks = store_weeks.sort_values('_TOTAL', ascending=False).drop(columns='_TOTAL')

print('\n── Weeks fired per store ──')
display(store_weeks)

Stores with at least one signal: 119

── Signals per store (total across all run dates) ──


C:\Users\djt20\AppData\Local\Temp\ipykernel_26188\1836398826.py:22: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  store_wide = store_wide.sort_values('_TOTAL', ascending=False).drop(columns='_TOTAL')


SIGNAL_TYPE     OSA                   COMPLIANCE
TRIGGER_NAME REPEAT VS_CHAIN  WOW TICK_AND_FLICK
STORE_ID                                        
270              58      121  189              1
204              40      133  135              0
224              66      101  102              0
535              73       97   82              0
447              51       95   75              0
...             ...      ...  ...            ...
404               0        3    4              0
448               0        1    6              0
210               0        0    2              5
236               0        0    0              6
570               0        0    1              0

[119 rows x 4 columns]


── Weeks fired per store ──


C:\Users\djt20\AppData\Local\Temp\ipykernel_26188\1836398826.py:36: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  store_weeks = store_weeks.sort_values('_TOTAL', ascending=False).drop(columns='_TOTAL')


SIGNAL_TYPE     OSA                  COMPLIANCE
TRIGGER_NAME REPEAT VS_CHAIN WOW TICK_AND_FLICK
STORE_ID                                       
638              17       17  17              6
240              17       17  17              6
517              17       14  17              8
492              17       15  17              7
282              17       17  17              5
...             ...      ...  ..            ...
210               0        0   2              5
448               0        1   5              0
404               0        3   3              0
236               0        0   0              6
570               0        0   1              0

[119 rows x 4 columns]